# 02 — PDF Extraction with Qwen LLM (vLLM Accelerated) — v3

Extract structured budget data from OPP PDF documents using Qwen LLM with vLLM batch inference.

**v3 improvements** (multi-column extraction):
- Replaces single `monto` column with `credito_vigente`, `credito_ejecutado`, `inversion`
- Each amount goes to its proper column — fixes 3-6x inflation for incisos with multiple amount types
- At least one amount column must be non-null per row
- v3 checkpoint prefix (`checkpoints_v3/`) — does NOT mix with v1/v2 data

**v2 improvements** (data quality fixes from BigQuery analysis):
- Canonical inciso reference list (1-36, from SIIF/CGN) embedded in LLM prompt
- PDF filename parsing to provide expected inciso context per document
- Strict prompt rules: no percentages, KPI counts, equipment names, future years, or inciso=0
- Post-extraction: name correction to canonical names, inciso recovery from denominacion
- Multi-stage validation: basic validity → garbage categories → garbage denominacion → canonical names → dedup
- Corrected inciso 28 (BPS), 29 (ASSE), 30 (Deuda Publica) per official SIIF classifier

**Architecture:**
- **Auto-selects model** based on GPU: Qwen2.5-14B on A100 (>=40GB), 7B on L4/A10, 3B on T4
- **vLLM engine**: Continuous batching for maximum GPU utilization (~5-10x faster than sequential HF inference)
- **Parallel prefetch**: CPU extracts text from next PDF while GPU processes current one

**Runtime:** Google Colab with GPU (Runtime -> Change runtime type -> A100 recommended)

**Pipeline:**
1. Download PDFs from GCS
2. pdfplumber: extract all tables from all pages -> upload to GCS
3. Qwen LLM (vLLM): batch-extract budget data with multi-column schema + canonical reference -> upload to GCS
4. Multi-stage validation, canonical name correction, deduplicate
5. Upload final Parquet to GCS (overwrites budget_data.parquet)

## Setup

In [ ]:
!pip install -q polars==1.24.0 pyarrow==18.1.0 pdfplumber==0.11.6 \
    google-cloud-storage==2.19.0 vllm

In [ ]:
from google.colab import auth
auth.authenticate_user()

import io
import json
import re
import time
import multiprocessing
from pathlib import Path
from urllib.parse import unquote
from concurrent.futures import ThreadPoolExecutor

import polars as pl
import pdfplumber
import torch
from google.cloud import storage

# ── Configuration ──
PROJECT_ID  = "fabled-imagery-488015-p6"
BUCKET_NAME = "opp-data-lake-fabled-imagery-488015-p6"

gcs_client = storage.Client(project=PROJECT_ID)
bucket     = gcs_client.bucket(BUCKET_NAME)

PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)

N_WORKERS = min(multiprocessing.cpu_count(), 8)

def upload_df_to_gcs(df: pl.DataFrame, blob_name: str) -> None:
    """Write a DataFrame as Parquet and upload to GCS."""
    buf = io.BytesIO()
    df.write_parquet(buf)
    buf.seek(0)
    blob = bucket.blob(blob_name)
    blob.upload_from_file(buf, content_type="application/octet-stream")


# ── GPU detection & model selection ──
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

if vram_gb >= 40:
    MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
    model_label = "14B FP16"
elif vram_gb >= 16:
    MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
    model_label = "7B FP16"
else:
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
    model_label = "3B FP16"

print(f"Project:    {PROJECT_ID}")
print(f"Bucket:     {BUCKET_NAME}")
print(f"GPU:        {gpu_name} ({vram_gb:.1f} GB)")
print(f"Model:      {MODEL_ID} ({model_label})")
print(f"CPU cores:  {multiprocessing.cpu_count()} (using {N_WORKERS} workers)")


# ── Canonical inciso reference (from SIIF / CGN) ──
CANONICAL_INCISOS = {
    1: "Poder Legislativo",
    2: "Presidencia de la Republica",
    3: "Ministerio de Defensa Nacional",
    4: "Ministerio del Interior",
    5: "Ministerio de Economia y Finanzas",
    6: "Ministerio de Relaciones Exteriores",
    7: "Ministerio de Ganaderia, Agricultura y Pesca",
    8: "Ministerio de Industria, Energia y Mineria",
    9: "Ministerio de Turismo",
    10: "Ministerio de Transporte y Obras Publicas",
    11: "Ministerio de Educacion y Cultura",
    12: "Ministerio de Salud Publica",
    13: "Ministerio de Trabajo y Seguridad Social",
    14: "Ministerio de Vivienda y Ordenamiento Territorial",
    15: "Ministerio de Desarrollo Social",
    16: "Poder Judicial",
    17: "Tribunal de Cuentas",
    18: "Corte Electoral",
    19: "Tribunal de lo Contencioso Administrativo",
    20: "Intereses y Otros Gastos de la Deuda",
    21: "Subsidios y Subvenciones",
    22: "Transferencias Financieras al Sector Seguridad Social",
    23: "Partidas a Reaplicar",
    24: "Diversos Creditos",
    25: "Administracion Nacional de Educacion Publica",
    26: "Universidad de la Republica",
    27: "Instituto del Nino y Adolescente del Uruguay",
    28: "Banco de Prevision Social",
    29: "Administracion de los Servicios de Salud del Estado",
    30: "Deuda Publica",
    31: "Universidad Tecnologica del Uruguay",
    32: "Instituto Uruguayo de Meteorologia",
    33: "Fiscalia General de la Nacion",
    34: "Junta de Transparencia y Etica Publica",
    35: "Instituto Nacional de Inclusion Social Adolescente",
    36: "Ministerio de Ambiente",
}

# Build a compact reference string for the LLM prompt
INCISO_REF_STR = "\n".join(
    f"  {k}: {v}" for k, v in CANONICAL_INCISOS.items()
)

# Reverse lookup: lowercase name fragment -> inciso number
_NAME_TO_INCISO = {}
for inc, name in CANONICAL_INCISOS.items():
    _NAME_TO_INCISO[name.lower()] = inc
    # Also index short forms: first 3 significant words
    words = [w for w in name.lower().split() if w not in ("de", "del", "la", "las", "los", "y", "al")]
    if len(words) >= 2:
        _NAME_TO_INCISO[" ".join(words[:2])] = inc


def parse_inciso_range_from_filename(pdf_name: str) -> list[int]:
    """Extract expected inciso numbers from the PDF filename.

    Filenames often contain patterns like:
      "Incisos del 02 al 06, 21 y 24"
      "Incisos 11 a 15 y 36"
      "Inciso 11"
    """
    decoded = unquote(pdf_name)

    # Pattern: "Incisos del XX al YY" or "Incisos XX a YY"
    range_pattern = r'[Ii]ncisos?\s+(?:del?\s+)?(\d{1,2})\s+(?:al?)\s+(\d{1,2})'
    # Pattern: standalone inciso numbers after "Inciso(s)"
    list_pattern = r'[Ii]ncisos?[^.]*?(\d{1,2}(?:\s*[,y]\s*\d{1,2})*)'

    found = set()

    for m in re.finditer(range_pattern, decoded):
        lo, hi = int(m.group(1)), int(m.group(2))
        found.update(range(lo, hi + 1))

    for m in re.finditer(list_pattern, decoded):
        nums = re.findall(r'\d{1,2}', m.group(1))
        found.update(int(n) for n in nums if 1 <= int(n) <= 36)

    # Filter to valid range
    return sorted(n for n in found if 1 <= n <= 36)


# Quick test
_test = parse_inciso_range_from_filename(
    "Incisos%20del%2002%20al%2006%2C%2021%20y%2024.pdf"
)
print(f"Filename parser test: {_test}")  # Should be [2, 3, 4, 5, 6, 21, 24]

## 1. Download PDFs from GCS

In [ ]:
blobs = list(bucket.list_blobs(prefix="raw/pdfs/"))
pdf_blobs = [b for b in blobs if b.name.endswith(".pdf")]
print(f"Found {len(pdf_blobs)} PDFs in GCS")

pdf_files = []
for blob in pdf_blobs:
    local_path = PDF_DIR / blob.name.split("/")[-1]
    if not local_path.exists():
        blob.download_to_filename(str(local_path))
        print(f"  Downloaded: {local_path.name} ({blob.size / 1024:.0f} KB)")
    else:
        print(f"  Cached:     {local_path.name}")
    pdf_files.append(local_path)

print(f"\n{len(pdf_files)} PDFs ready for processing")

## 2. pdfplumber Extraction (all pages, all tables)

Extracts structured tables from every page. Results uploaded to GCS as Parquet.

In [ ]:
def extract_tables_pdfplumber(pdf_path: Path) -> list[dict]:
    """Extract all tables from a PDF, returning list of row dicts with metadata."""
    rows = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                for table in page.extract_tables():
                    if not table or len(table) < 2:
                        continue
                    headers = [
                        str(h).strip() if h else f"col_{i}"
                        for i, h in enumerate(table[0])
                    ]
                    seen = {}
                    unique_headers = []
                    for h in headers:
                        if h in seen:
                            seen[h] += 1
                            unique_headers.append(f"{h}_{seen[h]}")
                        else:
                            seen[h] = 0
                            unique_headers.append(h)

                    for row in table[1:]:
                        row_dict = {
                            h: (str(cell).strip() if cell else "")
                            for h, cell in zip(unique_headers, row)
                        }
                        row_dict["_pdf_source"] = pdf_path.name
                        row_dict["_page_number"] = page.page_number
                        row_dict["_extraction_method"] = "pdfplumber"
                        rows.append(row_dict)
    except Exception as e:
        print(f"  SKIP {pdf_path.name} (corrupted): {e}")
    return rows


# Check if pdfplumber results already exist in GCS
PLUMBER_BLOB = "processed/pdf_extractions/pdfplumber_tables.parquet"
plumber_cached = bucket.blob(PLUMBER_BLOB).exists()

if plumber_cached:
    print(f"pdfplumber results cached in GCS, loading...")
    blob = bucket.blob(PLUMBER_BLOB)
    buf = io.BytesIO()
    blob.download_to_file(buf)
    buf.seek(0)
    df_plumber = pl.read_parquet(buf)
    all_plumber_rows = df_plumber.to_dicts()
    print(f"Loaded {len(all_plumber_rows)} cached rows from GCS")
else:
    t0 = time.time()
    all_plumber_rows = []
    for pdf_path in pdf_files:
        rows = extract_tables_pdfplumber(pdf_path)
        all_plumber_rows.extend(rows)
        print(f"{pdf_path.name}: {len(rows)} rows extracted")

    elapsed = time.time() - t0
    print(f"\nTotal rows via pdfplumber: {len(all_plumber_rows)} ({elapsed:.1f}s)")

    if all_plumber_rows:
        df_plumber = pl.DataFrame(all_plumber_rows)
        upload_df_to_gcs(df_plumber, PLUMBER_BLOB)
        print(f"Uploaded pdfplumber results to GCS ({df_plumber.shape[0]} rows, {df_plumber.shape[1]} cols)")
    else:
        df_plumber = pl.DataFrame()
        print("No tables extracted via pdfplumber")

## 3. Check Qwen Cache or Load Model (vLLM)

Skips model loading entirely if Qwen results already exist in GCS.
Uses vLLM for continuous batching — processes all pages of a PDF in one GPU call.

In [ ]:
QWEN_BLOB = "processed/pdf_extractions/budget_data_v3_raw.parquet"
qwen_cached = bucket.blob(QWEN_BLOB).exists()

# Set to True to force resume from per-PDF checkpoints instead of
# loading the final aggregated file. Use this when you still have
# unprocessed PDFs and need to continue extraction.
FORCE_RESUME = True
if FORCE_RESUME:
    qwen_cached = False
    print("FORCE_RESUME enabled — will resume from per-PDF v3 checkpoints")

if qwen_cached:
    print("Qwen v3 results cached in GCS — skipping model load")
    print("Loading cached results in next cell...")
    llm = None
else:
    from vllm import LLM, SamplingParams

    print(f"Loading {MODEL_ID} with vLLM (FP16)...")
    llm = LLM(
        model=MODEL_ID,
        dtype="float16",
        max_model_len=8192,
        gpu_memory_utilization=0.90,
        trust_remote_code=True,
    )
    sampling_params = SamplingParams(max_tokens=2048, temperature=0.1)

    print(f"vLLM engine ready — batch inference enabled")
    print(f"Max sequence length: 8192 | GPU util target: 90%")

## 4. Qwen Extraction — Batched (All Pages per PDF)

Processes every page of every PDF using vLLM batch inference.
All pages of a single PDF are sent to the GPU in one call for maximum throughput.
CPU prefetches text from the next PDF while GPU processes the current one.

In [ ]:
from transformers import AutoTokenizer

# Lightweight tokenizer (CPU only) for chat template formatting
_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

EXTRACTION_PROMPT = """You are a data extraction assistant for Uruguayan government budget documents (Presupuesto Nacional / Rendicion de Cuentas).

OFFICIAL INCISO REFERENCE (use ONLY these):
{inciso_ref}

Extract budget data from the text below into a JSON array. Each item MUST have:
- "inciso" (integer 1-36): Match the agency to the reference list above. Use the inciso NUMBER, not a sub-unit number.
- "denominacion_inciso" (string): Use the EXACT official name from the reference list above.
- "categoria" (string): spending category or program (e.g. "Formacion Bruta de Capital", "Funcionamiento")
- "credito_vigente" (number or null): approved/active budget amount in pesos uruguayos. Labels: "Credito Vigente", "Credito Asignado", "Presupuesto", "Credito Aprobado". Convert "1.234.567,89" to 1234567.89
- "credito_ejecutado" (number or null): actual executed spending in pesos uruguayos. Labels: "Credito Ejecutado", "Ejecucion", "Obligado", "Gasto Ejecutado", "Gasto Pagado"
- "inversion" (number or null): capital expenditure in pesos uruguayos. Labels: "Inversion", "Formacion Bruta de Capital", "Formacion de Capital", "Inversiones"
- "fiscal_year" (integer): 4-digit fiscal year (2005-2024 only)

AMOUNT COLUMN RULES:
- At least ONE of credito_vigente, credito_ejecutado, inversion must be non-null per row
- If the document has a single amount column and the type is unclear, put the value in credito_vigente as default
- If a table has columns like "Vigente" and "Ejecutado", map them to credito_vigente and credito_ejecutado respectively
- Do NOT put the same amount in multiple columns — each value goes to exactly one column

CRITICAL RULES:
- Return ONLY a valid JSON array, no other text
- If no budget data found, return []
- Do NOT extract percentages (e.g. "95%" or "Porcentaje Ejecucion") as amounts
- Do NOT extract KPI counts (e.g. "99 exposiciones") as amounts — only monetary amounts in UYU
- Do NOT extract equipment names (e.g. "Computadoras", "Impresoras") as denominacion_inciso
- Do NOT extract department/city names (e.g. "MONTEVIDEO", "SALTO") as denominacion_inciso
- Do NOT use 0 as inciso — if you cannot determine the inciso, SKIP that row
- Do NOT extract rows labeled "Total" or summary/subtotal rows
- categoria should be a spending type, NOT a sub-department or equipment item name
- For projected/future years beyond 2024, SKIP those rows
{pdf_context}
Text:
{text}
"""


def format_prompt(text: str, pdf_name: str = "") -> str:
    """Format text into a chat-templated prompt for vLLM, with PDF context."""
    # Build PDF-specific context hint from filename
    pdf_context = ""
    if pdf_name:
        expected_incisos = parse_inciso_range_from_filename(pdf_name)
        if expected_incisos:
            names = [f"{i}: {CANONICAL_INCISOS.get(i, '?')}" for i in expected_incisos]
            pdf_context = (
                f"\nThis PDF covers incisos: {', '.join(str(i) for i in expected_incisos)}\n"
                f"Expected agencies: {'; '.join(names)}\n"
                f"Only extract data for these incisos from this document.\n"
            )

    prompt_text = EXTRACTION_PROMPT.format(
        inciso_ref=INCISO_REF_STR,
        text=text[:3500],  # Reduced from 4000 to fit the larger prompt
        pdf_context=pdf_context,
    )
    messages = [{"role": "user", "content": prompt_text}]
    return _tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def parse_json_response(response: str) -> list[dict]:
    """Parse JSON array from LLM response text."""
    try:
        start = response.find("[")
        end = response.rfind("]") + 1
        if start >= 0 and end > start:
            return json.loads(response[start:end])
    except json.JSONDecodeError:
        pass
    return []


def correct_inciso_name(record: dict) -> dict:
    """Post-process: replace LLM denominacion with canonical name based on inciso number."""
    inc = record.get("inciso")
    if isinstance(inc, (int, float)) and int(inc) in CANONICAL_INCISOS:
        record["denominacion_inciso"] = CANONICAL_INCISOS[int(inc)]
    return record


def try_fix_inciso_from_name(record: dict) -> dict:
    """If inciso is 0 or missing, try to infer it from denominacion_inciso."""
    inc = record.get("inciso", 0)
    if inc and 1 <= inc <= 36:
        return record  # Already valid

    denom = (record.get("denominacion_inciso") or "").lower().strip()
    if not denom:
        return record

    # Try exact match
    if denom in _NAME_TO_INCISO:
        record["inciso"] = _NAME_TO_INCISO[denom]
        record["denominacion_inciso"] = CANONICAL_INCISOS[record["inciso"]]
        return record

    # Try substring match against canonical names
    for name_key, inc_num in _NAME_TO_INCISO.items():
        if name_key in denom or denom in name_key:
            record["inciso"] = inc_num
            record["denominacion_inciso"] = CANONICAL_INCISOS[inc_num]
            return record

    return record


def migrate_monto_to_columns(record: dict) -> dict:
    """Backward compat: if record has old 'monto' field, migrate to credito_vigente."""
    if "monto" in record and "credito_vigente" not in record:
        record["credito_vigente"] = record.pop("monto")
    elif "monto" in record:
        record.pop("monto")
    # Ensure all three amount columns exist
    record.setdefault("credito_vigente", None)
    record.setdefault("credito_ejecutado", None)
    record.setdefault("inversion", None)
    return record


def extract_pages_batch(page_texts: list[tuple[int, str]], pdf_name: str) -> list[dict]:
    """Extract budget data from all pages in a single batched vLLM call."""
    valid = [(pnum, text) for pnum, text in page_texts
             if text and len(text.strip()) >= 50]
    if not valid:
        return []

    prompts = [format_prompt(text, pdf_name) for _, text in valid]
    outputs = llm.generate(prompts, sampling_params)

    all_records = []
    for (pnum, _), output in zip(valid, outputs):
        response = output.outputs[0].text
        records = parse_json_response(response)
        for r in records:
            r["pdf_source_file"] = pdf_name
            r["page_number"] = pnum
            # Post-process: try to fix inciso=0, then normalize name
            r = try_fix_inciso_from_name(r)
            r = correct_inciso_name(r)
            r = migrate_monto_to_columns(r)
        all_records.extend(records)
    return all_records


def extract_text_from_pdf(pdf_path: Path) -> tuple[list[tuple[int, str]], int]:
    """Extract text from all pages of a PDF (CPU-bound, used for prefetch)."""
    pages = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                pages.append((page.page_number, page.extract_text() or ""))
    except Exception as e:
        print(f"  WARNING: Could not read {pdf_path.name}: {e}")
    return pages, len(pages)


print(f"Extraction functions ready (vLLM batch mode, v3 multi-column prompt)")
print(f"Canonical incisos loaded: {len(CANONICAL_INCISOS)} entries")
print(f"Output columns: credito_vigente, credito_ejecutado, inversion (replaces monto)")

In [ ]:
CHECKPOINT_PREFIX = "processed/pdf_extractions/checkpoints_v3/"

def get_completed_pdfs() -> set:
    """Check which PDFs already have checkpoint files in GCS."""
    blobs = list(bucket.list_blobs(prefix=CHECKPOINT_PREFIX))
    return {b.name.split("/")[-1].replace(".parquet", "") for b in blobs if b.name.endswith(".parquet")}

def save_checkpoint(pdf_name: str, records: list[dict]) -> None:
    """Save extraction results for a single PDF to GCS."""
    if not records:
        blob = bucket.blob(f"{CHECKPOINT_PREFIX}{pdf_name}.parquet")
        df_empty = pl.DataFrame({"_marker": ["empty"]})
        buf = io.BytesIO()
        df_empty.write_parquet(buf)
        buf.seek(0)
        blob.upload_from_file(buf, content_type="application/octet-stream")
        return
    df = pl.DataFrame(records, infer_schema_length=None)
    buf = io.BytesIO()
    df.write_parquet(buf)
    buf.seek(0)
    blob = bucket.blob(f"{CHECKPOINT_PREFIX}{pdf_name}.parquet")
    blob.upload_from_file(buf, content_type="application/octet-stream")


def load_checkpoint(pdf_name: str) -> list[dict]:
    """Load a checkpoint parquet from GCS, handling schema inconsistencies."""
    blob = bucket.blob(f"{CHECKPOINT_PREFIX}{pdf_name}.parquet")
    buf = io.BytesIO()
    blob.download_to_file(buf)
    buf.seek(0)

    for attempt, kwargs in enumerate([{}, {"use_pyarrow": True}]):
        try:
            if attempt > 0:
                buf.seek(0)
            df_chk = pl.read_parquet(buf, **kwargs)
            if "_marker" in df_chk.columns:
                return []
            return df_chk.to_dicts()
        except Exception:
            continue

    # Both readers failed (e.g. Int128/FixedLenByteArray) — cast to string
    try:
        import pyarrow.parquet as pq
        import pyarrow as pa
        buf.seek(0)
        table = pq.read_table(buf)
        new_cols = []
        for col in table.columns:
            try:
                _ = col.to_pylist()
                new_cols.append(col)
            except Exception:
                new_cols.append(col.cast(pa.string()))
        table2 = pa.table({name: col for name, col in zip(table.column_names, new_cols)})
        df_chk = pl.from_arrow(table2)
        if "_marker" in df_chk.columns:
            return []
        return df_chk.to_dicts()
    except Exception as e:
        print(f"  WARNING: Could not load checkpoint '{pdf_name}', skipping: {e}")
        return []


if qwen_cached:
    blob = bucket.blob(QWEN_BLOB)
    buf = io.BytesIO()
    blob.download_to_file(buf)
    buf.seek(0)
    df_cached = pl.read_parquet(buf)
    all_extractions = df_cached.to_dicts()
    total_pages = 0
    skipped_pdfs = []
    print(f"Loaded {len(all_extractions)} cached Qwen v3 records from GCS")
else:
    completed = get_completed_pdfs()
    if completed:
        print(f"Resuming: {len(completed)} PDFs already processed (v3), skipping them")

    all_extractions = []
    total_pages = 0
    skipped_pdfs = []
    pdfs_done_this_run = 0

    # Load existing checkpoint data (handles schema inconsistencies)
    loaded_count = 0
    skipped_checkpoints = 0
    for pdf_name in completed:
        records = load_checkpoint(pdf_name)
        if records:
            all_extractions.extend(records)
            loaded_count += 1
        else:
            skipped_checkpoints += 1

    if all_extractions:
        print(f"Loaded {len(all_extractions)} records from {loaded_count} v3 checkpoints")
    if skipped_checkpoints:
        print(f"Skipped {skipped_checkpoints} empty/unreadable checkpoints")

    # Build list of remaining PDFs
    pdfs_to_process = [p for p in pdf_files if p.stem not in completed]
    print(f"\nProcessing {len(pdfs_to_process)} remaining PDFs "
          f"(skipping {len(completed)} cached)")

    t0 = time.time()

    # Main loop with CPU prefetch + GPU batch inference
    prefetch_future = None
    with ThreadPoolExecutor(max_workers=2) as prefetch_pool:
        for pdf_idx, pdf_path in enumerate(pdfs_to_process):
            global_idx = pdf_files.index(pdf_path) + 1

            # Get page texts (from prefetch or extract now)
            if prefetch_future is not None:
                page_texts, n_pages = prefetch_future.result()
            else:
                page_texts, n_pages = extract_text_from_pdf(pdf_path)

            # Prefetch next PDF text on CPU while GPU processes current
            if pdf_idx + 1 < len(pdfs_to_process):
                prefetch_future = prefetch_pool.submit(
                    extract_text_from_pdf, pdfs_to_process[pdf_idx + 1]
                )
            else:
                prefetch_future = None

            if not page_texts:
                skipped_pdfs.append(pdf_path.name)
                print(f"[{global_idx}/{len(pdf_files)}] {pdf_path.name} — SKIP (no text)")
                continue

            print(f"[{global_idx}/{len(pdf_files)}] {pdf_path.name} "
                  f"({n_pages} pages, batched)")

            # Batch inference: ALL pages sent to vLLM in one call
            pdf_records = extract_pages_batch(page_texts, pdf_path.name)
            total_pages += n_pages

            # Checkpoint to GCS (async via thread)
            save_checkpoint(pdf_path.stem, pdf_records)
            all_extractions.extend(pdf_records)
            pdfs_done_this_run += 1

            elapsed = time.time() - t0
            rate = total_pages / elapsed if elapsed > 0 else 0
            print(f"  -> {len(pdf_records)} records | "
                  f"{rate:.1f} pages/sec | {elapsed:.0f}s elapsed")

    elapsed = time.time() - t0
    print(f"\nQwen v3 extraction complete:")
    print(f"  PDFs this run:   {pdfs_done_this_run}")
    print(f"  PDFs from cache: {len(completed)}")
    print(f"  Total records:   {len(all_extractions)}")
    print(f"  Pages this run:  {total_pages}")
    if total_pages > 0:
        print(f"  Time: {elapsed:.0f}s ({total_pages/elapsed:.1f} pages/sec)")
    if skipped_pdfs:
        print(f"  Skipped: {skipped_pdfs}")

## 5. Validate, Deduplicate & Combine

In [ ]:
EXPECTED_COLS = [
    "inciso", "denominacion_inciso", "categoria",
    "credito_vigente", "credito_ejecutado", "inversion",
    "fiscal_year", "pdf_source_file", "page_number",
]

# Categories that are clearly NOT spending categories (summaries, labels, metrics)
GARBAGE_CATEGORIES = {
    "total", "subtotal", "gran total", "total general",
    "credito asignado", "credito vigente", "credito ejecutado",
    "porcentaje ejecucion", "% ejecucion", "% ejecutado",
    "001 - direccion general de secretaria",
}

# Denominacion patterns that indicate equipment/item names, not agencies
GARBAGE_DENOM_PATTERNS = [
    r"^(tic|hardware|software|equipamiento|maquinaria|consultor|obras civiles|vehiculo)",
    r"^(montevideo|salto|tacuarembo|soriano|florida|rocha|rivera|flores|durazno|lavalleja|treinta|rio negro|san jose|maldonado|cerro largo|canelones|colonia|paysandu|artigas)(\s|$)",
    r"(unidad|impresora|computador|servidor|monitor|freezer|microondas|perchero)$",
    r"^centros de reclusion",
    r"^(kit |reloj |motor |taladro |sierra |torno |sillon )",
]

if all_extractions:
    # Normalize: keep only expected columns, fill missing with None
    # Handle backward compat: migrate old 'monto' field if present
    safe_records = []
    for r in all_extractions:
        rec = migrate_monto_to_columns(dict(r))
        safe_records.append({
            col: str(rec[col]) if col in rec and rec[col] is not None else None
            for col in EXPECTED_COLS
        })

    schema = {col: pl.Utf8 for col in EXPECTED_COLS}
    df_raw = pl.DataFrame(safe_records, schema=schema)
    print(f"Raw Qwen v3 records: {df_raw.shape[0]}")
    print(f"Columns: {df_raw.columns}")
    print(f"\nSample:")
    print(df_raw.head(5))

    # ── STAGE 0: Cast numeric columns ──
    df_clean = df_raw.with_columns([
        pl.col("inciso").cast(pl.Int64, strict=False),
        pl.col("credito_vigente").cast(pl.Float64, strict=False),
        pl.col("credito_ejecutado").cast(pl.Float64, strict=False),
        pl.col("inversion").cast(pl.Float64, strict=False),
        pl.col("fiscal_year").cast(pl.Int64, strict=False),
        pl.col("page_number").cast(pl.Int64, strict=False),
    ])

    # ── STAGE 1: Basic validity ──
    # At least one amount column must be >= 1000 and < 1e15
    has_valid_amount = (
        (pl.col("credito_vigente").is_not_null() & (pl.col("credito_vigente") >= 1000) & (pl.col("credito_vigente") < 1e15))
        | (pl.col("credito_ejecutado").is_not_null() & (pl.col("credito_ejecutado") >= 1000) & (pl.col("credito_ejecutado") < 1e15))
        | (pl.col("inversion").is_not_null() & (pl.col("inversion") >= 1000) & (pl.col("inversion") < 1e15))
    )

    df_valid = df_clean.filter(
        pl.col("inciso").is_not_null()
        & (pl.col("inciso") >= 1)
        & (pl.col("inciso") <= 36)
        & pl.col("fiscal_year").is_not_null()
        & (pl.col("fiscal_year") >= 2005)
        & (pl.col("fiscal_year") <= 2024)
        & has_valid_amount
    )
    stage1_count = df_valid.shape[0]
    print(f"\nStage 1 (basic validity): {df_raw.shape[0]} -> {stage1_count}")

    # ── STAGE 2: Filter garbage categories ──
    def is_garbage_category(cat: str) -> bool:
        if not cat:
            return False
        return cat.lower().strip() in GARBAGE_CATEGORIES

    df_valid = df_valid.filter(
        ~pl.col("categoria").map_elements(is_garbage_category, return_dtype=pl.Boolean)
    )
    stage2_count = df_valid.shape[0]
    print(f"Stage 2 (garbage categories): {stage1_count} -> {stage2_count}")

    # ── STAGE 3: Filter garbage denominacion_inciso ──
    import re as re_mod

    _garbage_re = [re_mod.compile(p, re_mod.IGNORECASE) for p in GARBAGE_DENOM_PATTERNS]

    def is_garbage_denom(denom: str) -> bool:
        if not denom or len(denom) <= 3:
            return True
        for pattern in _garbage_re:
            if pattern.search(denom):
                return True
        return False

    df_valid = df_valid.filter(
        ~pl.col("denominacion_inciso").map_elements(is_garbage_denom, return_dtype=pl.Boolean)
    )
    stage3_count = df_valid.shape[0]
    print(f"Stage 3 (garbage denominacion): {stage2_count} -> {stage3_count}")

    # ── STAGE 4: Force canonical denominacion_inciso ──
    def force_canonical_name(row):
        inc = row["inciso"]
        if inc and int(inc) in CANONICAL_INCISOS:
            return CANONICAL_INCISOS[int(inc)]
        return row["denominacion_inciso"]

    df_valid = df_valid.with_columns(
        pl.struct(["inciso", "denominacion_inciso"])
        .map_elements(force_canonical_name, return_dtype=pl.Utf8)
        .alias("denominacion_inciso")
    )

    # ── STAGE 5: Deduplicate ──
    before_dedup = df_valid.shape[0]
    df_valid = df_valid.unique(
        subset=["inciso", "credito_vigente", "credito_ejecutado", "inversion",
                "fiscal_year", "pdf_source_file"],
        keep="first",
    )

    print(f"Stage 5 (dedup): {before_dedup} -> {df_valid.shape[0]}")

    # ── Summary ──
    dropped = df_raw.shape[0] - df_valid.shape[0]
    pct_kept = df_valid.shape[0] / df_raw.shape[0] * 100 if df_raw.shape[0] > 0 else 0
    print(f"\n{'='*60}")
    print(f"VALIDATION SUMMARY (v3 — multi-column)")
    print(f"{'='*60}")
    print(f"  Raw records:     {df_raw.shape[0]}")
    print(f"  Valid records:   {df_valid.shape[0]} ({pct_kept:.1f}% kept)")
    print(f"  Dropped:         {dropped}")

    # Amount column coverage
    n_cv = df_valid.filter(pl.col("credito_vigente").is_not_null() & (pl.col("credito_vigente") > 0)).shape[0]
    n_ce = df_valid.filter(pl.col("credito_ejecutado").is_not_null() & (pl.col("credito_ejecutado") > 0)).shape[0]
    n_inv = df_valid.filter(pl.col("inversion").is_not_null() & (pl.col("inversion") > 0)).shape[0]
    print(f"\nAmount column coverage:")
    print(f"  credito_vigente:  {n_cv} rows ({n_cv/df_valid.shape[0]*100:.1f}%)")
    print(f"  credito_ejecutado: {n_ce} rows ({n_ce/df_valid.shape[0]*100:.1f}%)")
    print(f"  inversion:        {n_inv} rows ({n_inv/df_valid.shape[0]*100:.1f}%)")

    print(f"\nFiscal year distribution:")
    print(df_valid.group_by("fiscal_year").len().sort("fiscal_year"))
    print(f"\nInciso distribution (top 15):")
    print(df_valid.group_by("inciso", "denominacion_inciso").len()
          .sort("len", descending=True).head(15))
    print(f"\nRecords per PDF (top 10):")
    print(df_valid.group_by("pdf_source_file").len()
          .sort("len", descending=True).head(10))

    # Quick quality check: how many distinct denominacion per inciso (should be 1)
    denom_check = (df_valid.group_by("inciso")
                   .agg(pl.col("denominacion_inciso").n_unique().alias("n_names"))
                   .filter(pl.col("n_names") > 1))
    if denom_check.shape[0] > 0:
        print(f"\nWARNING: {denom_check.shape[0]} incisos still have multiple names:")
        print(denom_check)
    else:
        print(f"\nQuality check PASSED: all incisos have exactly 1 canonical name")
else:
    print("No records extracted. Check PDF content above.")
    df_valid = pl.DataFrame()

## 6. Upload to GCS

In [ ]:
# Upload Qwen v3 extraction results
if not df_valid.is_empty():
    upload_df_to_gcs(df_valid, "processed/pdf_extractions/budget_data.parquet")
    blob = bucket.blob("processed/pdf_extractions/budget_data.parquet")
    blob.reload()
    print(f"Uploaded {df_valid.shape[0]} v3 records to gs://{BUCKET_NAME}/processed/pdf_extractions/budget_data.parquet")
    print(f"File size: {blob.size / 1024:.0f} KB")
    print(f"Schema: {df_valid.columns}")
else:
    print("No valid records to upload.")

# Also upload the raw (pre-validation) data for debugging
if all_extractions:
    raw_safe = []
    for r in all_extractions:
        rec = migrate_monto_to_columns(dict(r))
        raw_safe.append({
            col: str(rec[col]) if col in rec and rec[col] is not None else None
            for col in EXPECTED_COLS
        })
    df_raw_out = pl.DataFrame(raw_safe, schema={col: pl.Utf8 for col in EXPECTED_COLS})
    upload_df_to_gcs(df_raw_out, "processed/pdf_extractions/budget_data_v3_raw.parquet")
    print(f"Uploaded {df_raw_out.shape[0]} raw v3 records for debugging")

In [ ]:
# Free GPU memory
if llm is not None:
    del llm
    if '_tokenizer' in dir():
        del _tokenizer
    torch.cuda.empty_cache()
    vram_after = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory freed. VRAM in use: {vram_after:.2f} GB")
else:
    print("Model was not loaded (cached results used)")

print(f"\nPDF extraction v2 complete.")
print(f"   Model:             {MODEL_ID} ({model_label})")
print(f"   pdfplumber rows:   {len(all_plumber_rows)}")
print(f"   Qwen v2 records:   {df_valid.shape[0] if not df_valid.is_empty() else 0}")
print(f"\nv2 improvements applied:")
print(f"   - Canonical inciso reference (36 entries) in prompt")
print(f"   - PDF filename context (expected inciso range)")
print(f"   - Strict rules: no percentages, KPIs, equipment names, future years")
print(f"   - Post-extraction name correction to canonical names")
print(f"   - Multi-stage validation with garbage filters")
print(f"\nGCS outputs:")
print(f"   processed/pdf_extractions/pdfplumber_tables.parquet")
print(f"   processed/pdf_extractions/budget_data.parquet (v2, overwrites v1)")
print(f"   processed/pdf_extractions/budget_data_v2_raw.parquet")